## Clinical Information Extraction with LLMs

This notebook demonstrates a pipeline for extracting structured clinical information
from emergency room (ER) reports using a large language model.

### Task

Extract:
- precipitating event (cause of ER visit)
- confidence
- explanation

### Pipeline

1. Clinical text extraction
2. Prompt-based information extraction
3. Structured JSON parsing

In [2]:
from transformers import pipeline
from pathlib import Path
import pandas as pd
import json

In [3]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

pipe = pipeline(
    "text-generation",
    model=MODEL_NAME,
    device_map="auto",
    max_new_tokens=512,
)

/home/tatiana.bladier/.local/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


In [4]:
def build_prompt(text: str) -> str:
    return f"""Identify the precipitating event that led to the medical emergency.

Return ONLY JSON:
- event
- confidence
- explanation

Medical text:
{text}
"""

In [5]:
def extract_medical_info(text: str) -> dict:
    prompt = build_prompt(text)

    output = pipe(prompt)[0]["generated_text"]

    return parse_json(output)

In [6]:
def parse_json(text: str):
    try:
        start = text.index("{")
        end = text.rindex("}") + 1
        return json.loads(text[start:end])
    except Exception:
        return {
            "event": None,
            "confidence": "unknown",
            "explanation": "Parsing failed",
            "raw_output": text
        }

In [7]:
def extract_clinical_text(raw_text: str) -> str:
    markers = [
        "CHIEF COMPLAINT:",
        "HISTORY OF PRESENT ILLNESS:",
        "REASON FOR VISIT:",
    ]

    lines = raw_text.splitlines()

    start_idx = next(
        (i for i, line in enumerate(lines)
         if any(line.strip().startswith(m) for m in markers)),
        None
    )

    if start_idx is None:
        return ""

    end_idx = next(
        (i for i, line in enumerate(lines)
         if "About This Sample:" in line),
        len(lines)
    )

    return "\n".join(l for l in lines[start_idx:end_idx] if l.strip())

In [8]:
def process_er_folder(folder_path: str, limit: int = 5) -> pd.DataFrame:
    records = []
    files = sorted(Path(folder_path).glob("*.txt"))[:limit]

    for file in files:
        raw = file.read_text(encoding="utf-8")
        text = extract_clinical_text(raw)

        if not text:
            continue

        result = extract_medical_info(text)
        result["source_file"] = file.name

        records.append(result)

    return pd.DataFrame(records)

In [10]:
df = process_er_folder("data/raw_data/mtsamples_er/texts", limit=5)
df

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


,event,confidence,explanation,raw_output,source_file
0,None,unknown,Parsing failed,Identify the precipitating event that led to t...,Abdominal_Pain_-_Consult.txt
1,None,unknown,Parsing failed,Identify the precipitating event that led to t...,Abdominal_Pain_2D_Consult.txt
2,None,unknown,Parsing failed,Identify the precipitating event that led to t...,Abrasions_26_Lacerations_2D_ER_Visit.txt
3,None,unknown,Parsing failed,Identify the precipitating event that led to t...,Abrasions__Lacerations_-_ER_Visit.txt
4,None,unknown,Parsing failed,Identify the precipitating event that led to t...,Accidental_Celesta_Ingestion_-_ER_Visit.txt


In [ ]:
df["event"].value_counts()

In [ ]:
for _, row in df.iterrows():
    print(f"--- {row['source_file']} ---")
    print(json.dumps(row.to_dict(), indent=2))

### Evaluation